 ## 🗂️🌌 **Preparing Data - Extracting Satellite Features from Combined Data (Landsat)**

<h3>Extracting Landsat Data Using API Calls</h3> <p align="justify"> The API-based method allows us to efficiently access <b>Landsat</b> data for specific coordinates and time periods, ensuring scalability and reproducibility of the process. </p> <p align="justify"> Through the API, we can query individual bands or compute indices like <b>NDMI</b> on-the-fly. This approach reduces storage requirements and simplifies data preprocessing, making it ideal for large-scale environmental and water quality analysis. </p>

<p>The <b>compute_Landsat_values</b> function extracts Landsat surface reflectance values for specific sampling locations using a 100 m focal buffer around each point. For each location:</p>

<ul>
  <li>A bounding box (bbox) is created around the latitude and longitude coordinates.</li>
  <li>The Microsoft Planetary Computer API is queried for Landsat-8 Level-2 surface reflectance imagery within the date range.</li>
  <li>The nearest low-cloud (<10% cloud cover) scene is selected, and the specified bands (<b>green</b>, <b>nir08</b>, <b>swir16</b>, <b>swir22</b>) are loaded.</li>
  <li>Median values of the pixels within the bounding box are computed to reduce the effect of noise or outliers.</li>
</ul>

<p><b>Why the buffer value is 0.00089831:</b></p>

<p>We want a ~100 m buffer around each point. At the equator, 1 degree ≈ 110 km. Therefore, the degree equivalent of 100 m is:</p>

<p style="text-align:center;">
  <em>buffer_deg = 100 m / 110,000 m/deg ≈ 0.00089831</em>
</p>

<p>This slightly adjusted value ensures that the buffer approximately matches the pixel resolution of Landsat imagery, capturing a ~100 m area around each sampling location.</p>


In [3]:
import warnings
warnings.filterwarnings("ignore")

# Data manipulation and analysis
import numpy as np
import pandas as pd

# Planetary Computer tools for STAC API access and authentication
import pystac_client
import planetary_computer as pc
from odc.stac import stac_load
from pystac.extensions.eo import EOExtension as eo

from glob import glob
from datetime import date
from tqdm import tqdm
import os

In [4]:
base_dir = base_dir = os.path.dirname(os.getcwd())

data_dir = os.path.join(base_dir, "data")
os.makedirs(data_dir, exist_ok=True)

landsat_data_dir = os.path.join(base_dir, "data", "Landsat")
os.makedirs(landsat_data_dir, exist_ok=True)

In [ ]:
Combined_River_Water_Quality_df = pd.read_csv(os.path.join(data_dir, "combined_river_quality_dataset.csv"))
Combined_River_Water_Quality_df.head(10)

,River,Date,Latitude,Longitude,EC,Temp_C,SS_mgL,pH,TP_mgL,NO2_N_mgL,NO3_N_mgL,DO_mgL,Al_mgL,Fe_mgL,Mn_mgL,Zn_mgL,Cu_mgL,Chl_a_mgL
0,Siu Lek Yuen Nullah,2000-01-17,22.383917,114.21025,14033.0,18.2,4.4,8.1,0.20,0.050,0.78,7.2,0.060,0.250,0.29,0.005,0.0005,NaN
1,Siu Lek Yuen Nullah,2000-02-21,22.383917,114.21025,13854.0,16.3,3.2,7.7,0.15,0.040,0.67,7.0,0.060,0.200,0.22,0.010,0.0030,NaN
2,Siu Lek Yuen Nullah,2000-03-22,22.383917,114.21025,22878.0,22.2,3.2,7.7,0.23,0.047,0.64,6.8,0.070,0.410,0.15,0.040,0.0070,NaN
3,Siu Lek Yuen Nullah,2000-04-20,22.383917,114.21025,13533.0,25.5,5.6,8.0,0.09,0.022,0.35,8.3,0.170,0.140,0.09,0.010,0.0005,NaN
4,Siu Lek Yuen Nullah,2000-05-15,22.383917,114.21025,14684.0,26.5,10.0,8.5,0.09,0.010,0.27,9.0,0.120,0.120,0.07,0.020,0.0005,NaN
5,Siu Lek Yuen Nullah,2000-06-09,22.383917,114.21025,31035.0,26.9,6.2,7.8,0.13,0.022,0.28,5.9,0.025,0.025,0.09,0.005,0.0010,NaN
6,Siu Lek Yuen Nullah,2000-07-10,22.383917,114.21025,25513.0,29.6,5.0,8.3,0.09,0.017,0.35,9.9,0.120,0.140,0.14,0.030,0.0030,NaN
7,Siu Lek Yuen Nullah,2000-08-16,22.383917,114.21025,26645.0,27.5,4.6,8.0,0.10,0.013,0.40,6.5,0.130,0.130,0.14,0.020,0.0060,NaN
8,Siu Lek Yuen Nullah,2000-09-11,22.383917,114.21025,8921.0,27.0,3.3,7.9,0.10,0.024,0.51,7.7,0.140,0.190,0.11,0.020,0.0020,NaN
9,Siu Lek Yuen Nullah,2000-10-16,22.383917,114.21025,17547.0,24.5,3.4,7.0,0.12,0.036,0.48,7.5,0.080,0.130,0.13,0.020,0.0040,NaN


In [ ]:
Reduced_Combined_River_Water_Quality_df = Combined_River_Water_Quality_df.copy()

# Reduces duplicate API calls
Reduced_Combined_River_Water_Quality_df["lat_round"] = Reduced_Combined_River_Water_Quality_df["Latitude"].round(3)
Reduced_Combined_River_Water_Quality_df["lon_round"] = Reduced_Combined_River_Water_Quality_df["Longitude"].round(3)

Reduced_Combined_River_Water_Quality_df = Reduced_Combined_River_Water_Quality_df.drop_duplicates(subset=["lat_round", "lon_round", "Date"]).reset_index(drop=True)

In [ ]:
Reduced_Combined_River_Water_Quality_df.info()

In [ ]:
# Function to compute values for specified bands
# Setup
tqdm.pandas()

def compute_Landsat_values(row, required_bands):
    """
    Computes values for the specified bands
    """
    lat = row['Latitude']
    lon = row['Longitude']
    date = pd.to_datetime(row['Date'], dayfirst=True, errors='coerce')

    # Return NaN if date is invalid
    if pd.isna(date):
        return pd.Series({band: np.nan for band in required_bands})

    # Buffer size for ~100m 
    bbox_size = 0.00089831  
    bbox = [
        lon - bbox_size / 2,
        lat - bbox_size / 2,
        lon + bbox_size / 2,
        lat + bbox_size / 2
    ]

    try:
        catalog = pystac_client.Client.open(
            "https://planetarycomputer.microsoft.com/api/stac/v1",
            modifier=pc.sign_inplace,
        )

        # Wider search range, we'll filter to nearest date later
        search = catalog.search(
            collections=["landsat-c2-l2"],
            bbox=bbox,
            datetime="2000-01-01/2025-12-31",
            query={"eo:cloud_cover": {"lt": 10}},
        )
        
        items = search.item_collection()

        if not items:
            return pd.Series({band: np.nan for band in required_bands})

        # Convert sample date to UTC
        sample_date_utc = date.tz_localize("UTC") if date.tzinfo is None else date.tz_convert("UTC")

        # Pick the item closest to the sample date
        items = sorted(
            items,
            key=lambda x: abs(pd.to_datetime(x.properties["datetime"]).tz_convert("UTC") - sample_date_utc)
        )
        selected_item = pc.sign(items[0])

        # Load required bands
        data = stac_load([selected_item], bands=required_bands, bbox=bbox).isel(time=0)

        results = {}

        # Compute medians for each band
        for band in required_bands:
            band_data =  data[band].astype("float")

            # Compute medians
            median_value = float(band_data.median(skipna=True).values)

            # Replace 0 with NaN
            median_value = median_value if median_value != 0 else np.nan
            results[band] = median_value

        return pd.Series(results)
    
    except Exception as e:
            print(f"Error processing row: {e}")
            return pd.Series({band: np.nan for band in required_bands})


In [7]:
landsat_csv_files = sorted(glob(os.path.join(landsat_data_dir, "Combined_River_Water_Quality_batch_*.csv")))

In [ ]:
# First time or continue run from last batch
tqdm.pandas()

batch_size = 200

# Define required bands
required_bands = ["blue", "red", "green", "nir", "swir16", "swir22"]

# Completed batches
completed_batches = []

for f in landsat_csv_files:
    try:
        batch_num = int(os.path.basename(f).split("_")[-1].replace(".csv", ""))
        completed_batches.append(batch_num)
    except:
        pass

# Determine starting index based on completed batches
if completed_batches:
    last_batch = max(completed_batches)
    start_idx = last_batch + batch_size

    print(f"Last completed batch: {last_batch}. Starting next batch from index: {start_idx}")

else:
    start_idx = 0
    print("No completed batches found. Starting from index 0.")

# Store processed batches
batch_results = []

# Process batches
for idx in range(start_idx, len(Reduced_Combined_River_Water_Quality_df), batch_size):
    batch = Reduced_Combined_River_Water_Quality_df.iloc[idx:idx + batch_size].copy()

    print(f"Processing batch {idx} to {idx + len(batch)}...")
    try:
        # Extract features
        features = batch.progress_apply(lambda row: compute_Landsat_values(row, required_bands), axis=1)
        
        # Append features to batch
        batch = pd.concat([batch.reset_index(drop=True), features.reset_index(drop=True)], axis=1)

        batch_results.append(batch)

        # Save progress
        output_path = os.path.join(landsat_data_dir, f"Combined_River_Water_Quality_batch_{idx}.csv")
        batch.to_csv(output_path, index=False)
        print(f"Batch {idx} saved to {output_path}")

    except Exception as e:
        print(f"Error processing batch {idx}: {e}")
        continue    

print("All batches processed successfully")

In [ ]:
# Define required bands
required_bands = ["blue", "red", "green", "nir", "swir16", "swir22"]

# Update processed files with newly added bands 
for file_path in landsat_csv_files:

    print(f"\nUpdating file: {os.path.basename(file_path)}")

    # Load existing batch file
    df = pd.read_csv(file_path)

    # Skip already updated files
    missing_bands = [band for band in required_bands if band not in df.columns]

    if not missing_bands:
        print("Bands already present. File already updated, skipping...")
        continue

    print(f"Missing bands: {missing_bands}. Computing values for these bands...")

    # Extract only new bands
    new_features = df.progress_apply(lambda row: compute_Landsat_values(row, missing_bands), axis=1)

    # Append new bands to existing DataFrame
    df = pd.concat([df, new_features], axis=1)

    # Overwrite existing CSV
    df.to_csv(file_path, index=False)

    print(f"Updated file saved: {os.path.basename(file_path)}")

print("\n All Landsat batch files updated successfully with new bands.")


Updating file: Combined_River_Water_Quality_batch_0.csv
Bands already present. File already updated, skipping...

Updating file: Combined_River_Water_Quality_batch_1000.csv
Bands already present. File already updated, skipping...

Updating file: Combined_River_Water_Quality_batch_10000.csv
Bands already present. File already updated, skipping...

Updating file: Combined_River_Water_Quality_batch_10200.csv
Bands already present. File already updated, skipping...

Updating file: Combined_River_Water_Quality_batch_10400.csv
Bands already present. File already updated, skipping...

Updating file: Combined_River_Water_Quality_batch_10600.csv
Bands already present. File already updated, skipping...

Updating file: Combined_River_Water_Quality_batch_10800.csv
Bands already present. File already updated, skipping...

Updating file: Combined_River_Water_Quality_batch_11000.csv
Bands already present. File already updated, skipping...

Updating file: Combined_River_Water_Quality_batch_11200.csv


100%|██████████| 200/200 [16:37<00:00,  4.99s/it]


Updated file saved: Combined_River_Water_Quality_batch_6800.csv

Updating file: Combined_River_Water_Quality_batch_7000.csv


100%|██████████| 200/200 [19:24<00:00,  5.82s/it]


Updated file saved: Combined_River_Water_Quality_batch_7000.csv

Updating file: Combined_River_Water_Quality_batch_7200.csv


...

Updated file saved: Combined_River_Water_Quality_batch_9400.csv

Updating file: Combined_River_Water_Quality_batch_9600.csv


100%|██████████| 200/200 [18:44<00:00,  5.62s/it]


Updated file saved: Combined_River_Water_Quality_batch_9600.csv

Updating file: Combined_River_Water_Quality_batch_9800.csv


100%|██████████| 200/200 [18:11<00:00,  5.46s/it]

Updated file saved: Combined_River_Water_Quality_batch_9800.csv

 All Landsat batch files updated successfully with new bands.


In [ ]:
# Get all batch files
landsat_csv_files = sorted(glob(os.path.join(landsat_data_dir, "Combined_River_Water_Quality_batch_*.csv")))

print(f"Found {len(landsat_csv_files)} batch files. Combining...")

# Read and combine all batches
landsat_df_list = []

for file in landsat_csv_files:
    print(f"Reading: {os.path.basename(file)}")
    df = pd.read_csv(file)
    landsat_df_list.append(df)

# Concatenate all batch files into a single DataFrame
Combined_River_Water_Quality_Landsat_df = pd.concat(landsat_df_list, ignore_index=True)

print(f"\n Final combined DataFrame shape: {Combined_River_Water_Quality_Landsat_df.shape}")

# Save combined file
output_path = os.path.join(data_dir, "Combined_River_Water_Quality_Final.csv")
Combined_River_Water_Quality_Landsat_df.to_csv(output_path, index=False)

print(f"\n Combined dataset saved to: {output_path}")

Found 178 batch files. Combining...
Reading: Combined_River_Water_Quality_batch_0.csv
Reading: Combined_River_Water_Quality_batch_1000.csv
Reading: Combined_River_Water_Quality_batch_10000.csv
Reading: Combined_River_Water_Quality_batch_10200.csv
Reading: Combined_River_Water_Quality_batch_10400.csv
Reading: Combined_River_Water_Quality_batch_10600.csv
Reading: Combined_River_Water_Quality_batch_10800.csv
Reading: Combined_River_Water_Quality_batch_11000.csv
Reading: Combined_River_Water_Quality_batch_11200.csv
Reading: Combined_River_Water_Quality_batch_11400.csv
Reading: Combined_River_Water_Quality_batch_11600.csv
Reading: Combined_River_Water_Quality_batch_11800.csv
Reading: Combined_River_Water_Quality_batch_1200.csv
Reading: Combined_River_Water_Quality_batch_12000.csv
Reading: Combined_River_Water_Quality_batch_12200.csv
Reading: Combined_River_Water_Quality_batch_12400.csv
Reading: Combined_River_Water_Quality_batch_12600.csv
Reading: Combined_River_Water_Quality_batch_12800.cs